# **Structured Outputs** (Groq Version)

By default the LLM returns free-form text. Structured Outputs force it to return
a specific shape — great for downstream code that needs to parse the result.

In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

# temperature=0 → deterministic; important for structured output reliability
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


Bro API KEY Variable exists


# **Guiding in Prompts**

The simplest approach: just ask the LLM *in the prompt* to use a certain format.
Works but is fragile — the model may not always follow.

In [2]:
from langchain_core.prompts import PromptTemplate

# We're instructing the LLM about the output format INSIDE the prompt text itself
# The model will TRY to follow it but there's no schema enforcement
result = llm_groq.invoke(
    "Tell me a joke. Generate the output in key-value pair format with the following keys: setup, punchline"
)
result.content  # Returns plain text — no guarantee of exact structure


'{\n  "setup": "Why couldn\'t the bicycle stand up by itself?",\n  "punchline": "Because it was two-tired."\n}'

# **Using Pydantic Models**

Pydantic enforces the schema at the Python level AND instructs the LLM via
the model's function-calling mechanism — much more reliable than prompt instructions.

In [7]:
from pydantic import BaseModel, Field

# BaseModel is Pydantic's base class for data schemas
# Each field defines: name, type, and an optional description
# The description is passed to the LLM so it knows what to put in each field
class llm_schema(BaseModel):
    setup:    str = Field(description="The setup for the joke")
    punchline: str = Field(description="The punchline for the joke")


In [4]:
# .with_structured_output() wraps the LLM with a schema enforcer
# Under the hood it uses the LLM's tool/function-calling to guarantee JSON output
# that matches the Pydantic model — then parses + validates it automatically
llm_structured_output = llm_groq.with_structured_output(llm_schema)

# The return value is now an llm_schema INSTANCE, not a raw string
result = llm_structured_output.invoke("Tell me a joke")

# Access fields like a normal Python object — type-safe!
result.punchline


'Because it was two-tired'

# **Using TypedDict**

TypedDict is Python's stdlib alternative to Pydantic. Less feature-rich but no extra
dependency. The result comes back as a plain dict instead of a class instance.

In [5]:
from typing import TypedDict

# TypedDict defines the expected keys and their types — like a typed dict blueprint
# No Field() descriptions here — less context given to the LLM vs Pydantic
class llm_schema_td(TypedDict):
    setup:    str
    punchline: str


In [6]:
# Same API as Pydantic — .with_structured_output() works with TypedDict too
llm_structured_typed_dict = llm_groq.with_structured_output(llm_schema_td)

result = llm_structured_typed_dict.invoke("Tell me a joke")

# Result is now a plain dict: {'setup': '...', 'punchline': '...'}
# Access via result['setup'] or result['punchline']
result


{'punchline': 'Because it was two-tired',
 'setup': "Why couldn't the bicycle stand up by itself?"}